# Lab 14: Debugging and Traceability (Verification)

**Problem Statement:**
Write a trace program to step through a discrete-event model and log system states, ensuring it adheres exactly to the logical model rules.


### Theory and Approach

**Verification** in simulation ensures that the conceptual model has been correctly translated into a computer program. One of the most effective verification techniques is **Tracing**.

A **Trace** logs the state of the system, the event list, and statistical counters at each step (every time an event occurs). By manually checking this trace against hand-calculated values for a small number of events, we can verify the model's logic.

We will simulate a simple Single-Server Queue (like an M/M/1) but with deterministic or very simple random inputs, and output a detailed trace table.


In [ ]:
import pandas as pd

# ==========================================
# 1. Simple DES Model with Tracing
# ==========================================
# Let's use predefined arrival and service times to easily verify by hand.
arrival_times = [0, 2, 5, 6, 8] # Customers arrive at these exact times
service_times = [3, 4, 1, 2, 2] # Service durations for each customer

num_customers = len(arrival_times)

# State Variables
clock = 0
server_busy = False
queue = []
events = [] # Event List: list of tuples (time, event_type, customer_id)

# Initialize first arrival
events.append((arrival_times[0], 'Arrival', 0))

# Trace Log
trace = []

def log_state(event_time, event_type, cust_id):
    """Helper function to record the system state"""
    trace.append({
        'Clock': event_time,
        'Event': f"{event_type} (Cust {cust_id})",
        'Server Busy': 'Yes' if server_busy else 'No',
        'Queue Length': len(queue),
        'Pending Events': [(e[0], e[1][0], e[2]) for e in sorted(events)] # Just showing Time, Type initials, Cust
    })

# Simulation Loop
customer_idx = 0
while events:
    # Sort and pop the next event
    events.sort()
    current_time, event_type, cust_id = events.pop(0)
    clock = current_time
    
    if event_type == 'Arrival':
        if not server_busy:
            server_busy = True
            # Schedule Departure
            events.append((clock + service_times[cust_id], 'Departure', cust_id))
        else:
            queue.append(cust_id)
            
        # Schedule next arrival if there are more customers
        if cust_id + 1 < num_customers:
            events.append((arrival_times[cust_id + 1], 'Arrival', cust_id + 1))
            
    elif event_type == 'Departure':
        if len(queue) > 0:
            next_cust = queue.pop(0)
            # Schedule Departure for the next customer in queue
            events.append((clock + service_times[next_cust], 'Departure', next_cust))
        else:
            server_busy = False
            
    # Log the state AFTER processing the event
    log_state(clock, event_type, cust_id)

# ==========================================
# 2. Display Trace Table
# ==========================================
trace_df = pd.DataFrame(trace)
print("--- Simulation Trace Table (Verification) ---")
display(trace_df)
